# 00 - Setup and a very short Python refresher

**~5 minutes.** If you already use Python daily, just run the cells and move on to notebook 01.

By the end of this notebook you will have checked that F4Enix works on your machine, and you
will have seen the three Python idioms that the rest of the workshop relies on.

## 1. Where things are

Every notebook starts with the same cell. It only defines two folders:

* `DATA` - the MCNP files we work on. **Never edit anything in there.**
* `OUT` - where everything we produce is written. You can empty it and re-run at any time.

`Path` comes from the standard library and lets you build file paths with `/`, which works
identically on Windows and Linux.

In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

## 2. Is F4Enix installed?

F4Enix takes a few seconds to import the first time (it loads `pyvista`, the 3-D engine).

In [ ]:
import f4enix

print("F4Enix version:", f4enix.__version__)

The four objects you will use most are importable straight from the package. Each one wraps one
kind of MCNP file:

| Object | File it reads |
|--------|---------------|
| `Input` | an MCNP input (`.i`) |
| `Output` | an MCNP output (`outp`) |
| `Mctal` | a `mctal` file |
| `Meshtal` | a `meshtal` file |

In [ ]:
from f4enix import Input, Mctal, Meshtal, Output

print(Input, Output, Mctal, Meshtal, sep="\n")

## 3. Idiom 1 - a dictionary

F4Enix hands you cells, surfaces and data cards through objects that behave like a Python
**dictionary**: you look things up with square brackets, and you can ask for the list of keys.

Note that the keys are **strings**, even for cell numbers: `inp.cells["2"]`, not `inp.cells[2]`.

In [ ]:
# a plain Python dictionary, to fix the idea
densities = {"steel": 7.9, "water": 1.0, "copper": 8.9}

print("all the keys     :", list(densities.keys()))
print("one value        :", densities["water"])
print("is water there?  :", "water" in densities)

for name, value in densities.items():
    print(f"  {name:8s} -> {value} g/cc")

## 4. Idiom 2 - a pandas DataFrame

Many F4Enix methods return a `DataFrame`: a table with named columns and an index.
You need exactly four things from it.

In [ ]:
import pandas as pd

df = pd.DataFrame(
    {"material": [7, 11, 7, 13], "density": [1.0, 7.9, 1.0, 5.9]},
    index=[34, 35, 36, 37],
)
df.index.name = "cell"
df

In [ ]:
# (a) look at the first rows
df.head(2)

In [ ]:
# (b) take one column
df["material"]

In [ ]:
# (c) keep only the rows that satisfy a condition -> this is the one you will use all the time
df[df["material"] == 7]

In [ ]:
# (d) get the index of the selected rows back as a plain Python list
list(df[df["material"] == 7].index)

## 5. Idiom 3 - making a quick plot

`matplotlib` is used in notebook 03 to plot a spectrum. This is all you need.

In [ ]:
import matplotlib.pyplot as plt

energies = [0.1, 1.0, 5.0, 10.0, 14.0]
flux = [1.2e10, 8.4e9, 3.1e9, 9.0e8, 4.2e9]

fig, ax = plt.subplots(figsize=(5, 3))
ax.step(energies, flux, where="post")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Energy [MeV]")
ax.set_ylabel("Flux [n/cm2]")
ax.grid(alpha=0.3)

## 6. Final check

Run the cell below. If it prints `READY`, you are set for the workshop.

If it fails, run `python check_setup.py` from a terminal in the workshop folder: it gives a
much more detailed diagnosis.

In [ ]:
expected = [
    DATA / "iter_1d" / "ITER_1D.i",
    DATA / "iter_1d" / "outp",
    DATA / "iter_1d" / "mctal",
    DATA / "iter_1d" / "meshtal",
    DATA / "sphere" / "Sphere.i",
    DATA / "ww" / "ww_simple_cart",
]
missing = [p for p in expected if not p.is_file()]

if missing:
    print("MISSING FILES:")
    for p in missing:
        print("   ", p)
else:
    print("READY")